# Jack The Learner - Colab Training

**A humanoid robot brain that understands physics - built on transformers.**

## Setup
1. **Runtime → Change runtime type → T4 GPU** (or A100 for vision)
2. Run cells in order
3. Checkpoints save to Google Drive automatically

## Training Pipeline
```
Phase 0 (1-2 hrs)     Phase 1 (4-12 hrs)    Phase 2 (2-4 hrs)
Learn Physics    →    Learn Walking     →    Learn from Demos
MathReasoner          + WorldModel            ALL components
from SymPy            + HAC skills            refined
```

---

## 1. Setup Environment

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/JackTheLearner/checkpoints', exist_ok=True)
print('Google Drive mounted')

In [ ]:
# Install dependencies
!pip install -q torch torchvision
!pip install -q gymnasium mujoco
!pip install -q sympy tqdm tensorboard
print('Dependencies installed')

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU! Runtime > Change runtime type > T4 GPU')

In [ ]:
# Clone repository
%cd /content
!rm -rf JackTheLearner 2>/dev/null
!git clone https://github.com/JannoLouwrens/JackTheLearner.git
%cd JackTheLearner
!mkdir -p checkpoints
!ls *.py

---
## 2. Phase 0: Learn Physics (1-2 hours)

Neural network learns F=ma, torque, energy from SymPy ground truth.

In [ ]:
# Quick test (5 min)
!python Phase0_Physics.py --samples 1000 --epochs 5

In [ ]:
# Full Phase 0 (1-2 hours)
!python Phase0_Physics.py --samples 100000 --epochs 50

# Save to Drive
!cp checkpoints/phase0_*.pt /content/drive/MyDrive/JackTheLearner/checkpoints/ 2>/dev/null
print('Phase 0 complete!')

---
## 3. Phase 1: Learn Walking (4-12 hours)

RL training in MuJoCo. Auto-resumes from `rl_latest.pt` if exists.

In [ ]:
# Restore checkpoints from Drive (run this first if resuming)
import os

# Phase 0 checkpoint
if not os.path.exists('checkpoints/phase0_best.pt'):
    !cp /content/drive/MyDrive/JackTheLearner/checkpoints/phase0_best.pt checkpoints/ 2>/dev/null

# Phase 1 checkpoint (for resume)
!cp /content/drive/MyDrive/JackTheLearner/checkpoints/rl_latest.pt checkpoints/ 2>/dev/null
!cp /content/drive/MyDrive/JackTheLearner/checkpoints/rl_best.pt checkpoints/ 2>/dev/null

!ls -la checkpoints/

In [ ]:
# Phase 1: RL Walking
# Auto-resumes if checkpoints/rl_latest.pt exists!

!python Phase1_Locomotion.py \
    --phase0-checkpoint checkpoints/phase0_best.pt \
    --epochs 500

# Save to Drive
!cp checkpoints/rl_*.pt /content/drive/MyDrive/JackTheLearner/checkpoints/ 2>/dev/null
print('Phase 1 session complete!')

---
## 4. Phase 2: Learn from Demos (2-4 hours)

Imitation learning for natural movement.

In [ ]:
# Restore Phase 1 checkpoint
import os
if not os.path.exists('checkpoints/rl_best.pt'):
    !cp /content/drive/MyDrive/JackTheLearner/checkpoints/rl_best.pt checkpoints/ 2>/dev/null
    !cp /content/drive/MyDrive/JackTheLearner/checkpoints/rl_latest.pt checkpoints/ 2>/dev/null
!ls -la checkpoints/

In [ ]:
# Phase 2: Imitation Learning
!python Phase2_Imitation.py \
    --checkpoint-in checkpoints/rl_best.pt \
    --epochs 100

# Save to Drive
!cp checkpoints/phase2_*.pt /content/drive/MyDrive/JackTheLearner/checkpoints/
print('Phase 2 complete!')

---
## 5. Results

In [ ]:
# List checkpoints
print('Local:')
!ls -lh checkpoints/
print('\nDrive:')
!ls -lh /content/drive/MyDrive/JackTheLearner/checkpoints/

---
## Checkpoint Names

| Phase | Checkpoint | Description |
|-------|------------|-------------|
| 0 | `phase0_best.pt` | MathReasoner with physics |
| 1 | `rl_latest.pt` | Latest RL checkpoint (auto-resume) |
| 1 | `rl_best.pt` | Best RL checkpoint |
| 2 | `phase2_best.pt` | Final trained brain |

## Resume After Disconnect
1. Re-run cells 1-4 (setup)
2. Run "Restore checkpoints" cell
3. Run Phase 1 cell - it auto-resumes from `rl_latest.pt`

---
**Author:** Janno Louwrens